# 04 · Pipeline: entrenamiento y seguimiento con MLflow

Entrenamos tres modelos y los comparamos con la **validación**, no con test. Después evaluamos el mejor en **test**, una sola vez.

1. **Referencia sin aprender:** predice siempre la tasa base. Su AUC es 0,5. Si un modelo no la supera, no sirve.
2. **Regresión logística** (con imputación, logaritmo y escalado).
3. **Gradient boosting.**

Cada ejecución se registra en **MLflow**: parámetros, métricas y el modelo con su firma. Necesita ML construida (cuaderno 03).

In [ ]:
# Parámetros. None usa las rutas por defecto del proyecto (o las variables CURSO_DB, CURSO_MODELOS y CURSO_REPORTES).
RUTA_DB = None
RUTA_MODELOS = None
RUTA_REPORTES = None
USAR_MLFLOW = True
OBJETIVO = 'y'
FEATURES = ['recencia_dias', 'frecuencia', 'monto_total', 'ticket_medio', 'unidades_total', 'productos_distintos',
            'antiguedad_dias', 'frecuencia_90d', 'monto_90d', 'frecuencia_180d', 'dias_medios_entre_compras',
            'devoluciones', 'es_reino_unido']
VENTANA_OBJETIVO_DIAS = 90
CORTES = {
    'train': ['2010-06-01', '2010-07-01', '2010-08-01', '2010-09-01', '2010-10-01', '2010-11-01', '2010-12-01'],
    'valid': ['2011-03-01'],
    'test': ['2011-06-01', '2011-09-01'],
}

In [ ]:
import os, sqlite3, sys
from pathlib import Path

import pandas as pd

pd.set_option('display.width', 150, 'display.max_columns', 30)

# Carpeta del proyecto: la que contiene «sql» y «pyproject.toml».
RAIZ = Path.cwd().resolve()
while not ((RAIZ / 'sql').exists() and (RAIZ / 'pyproject.toml').exists()) and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
SQL = RAIZ / 'sql'
DB = Path(RUTA_DB or os.environ.get('CURSO_DB') or RAIZ / 'data' / 'processed' / 'curso.db')
DB.parent.mkdir(parents=True, exist_ok=True)

con = sqlite3.connect(DB)
con.execute('PRAGMA foreign_keys = ON')


def ejecutar_script(archivo):
    """Ejecuta un archivo .sql de la carpeta sql/ (puede tener varias sentencias)."""
    con.executescript((SQL / archivo).read_text(encoding='utf-8'))
    con.commit()


def escalar(consulta, parametros=None):
    fila = con.execute(consulta, parametros or {}).fetchone()
    return None if fila is None else fila[0]


print('Proyecto:', RAIZ)
print('Base de datos:', DB)
MODELOS = Path(RUTA_MODELOS or os.environ.get('CURSO_MODELOS') or RAIZ / 'models')
REPORTES = Path(RUTA_REPORTES or os.environ.get('CURSO_REPORTES') or RAIZ / 'reports')
(REPORTES / 'figures').mkdir(parents=True, exist_ok=True)
MODELOS.mkdir(parents=True, exist_ok=True)

## 1. Los datos: una tabla por partición
La partición viene de la capa ML. **No se mezclan filas entre particiones.**

In [ ]:
import hashlib, json, platform
from datetime import datetime

import joblib, numpy as np, sklearn

df = pd.read_sql('SELECT * FROM ml_dataset', con)
partes = {p: df[df['particion'] == p].reset_index(drop=True) for p in ('train', 'valid', 'test')}
X = {k: v[FEATURES] for k, v in partes.items()}
y = {k: v[OBJETIVO] for k, v in partes.items()}
for k in partes:
    assert len(partes[k]) > 0 and y[k].nunique() == 2, f'La partición {k} está vacía o tiene una sola clase: revisa los cortes.'
pd.DataFrame({'filas': {k: len(v) for k, v in partes.items()}, 'tasa_recompra': {k: round(v.mean(), 3) for k, v in y.items()}})

## 2. Los modelos y la evaluación

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler


def huella_datos(d):
    """Identifica con qué datos se entrenó el modelo."""
    h = hashlib.sha256(pd.util.hash_pandas_object(d.sort_values(['cliente_id', 'fecha_corte']), index=False).values.tobytes())
    return h.hexdigest()[:16]


def modelos():
    log1p = FunctionTransformer(np.log1p, feature_names_out='one-to-one')
    return {
        'referencia_sin_aprender': DummyClassifier(strategy='prior'),
        'regresion_logistica': Pipeline([('imputar', SimpleImputer(strategy='median')), ('log', log1p),
                                         ('escalar', StandardScaler()), ('modelo', LogisticRegression(max_iter=1000))]),
        'gradient_boosting': HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=0),
    }


def evaluar(modelo, Xp, yp):
    prob = modelo.predict_proba(Xp)[:, 1]
    corte10 = max(1, int(round(len(yp) * 0.10)))
    top = np.argsort(-prob)[:corte10]
    return {'auc': float(roc_auc_score(yp, prob)), 'precision_media': float(average_precision_score(yp, prob)),
            'tasa_base': float(np.mean(yp)), 'aciertos_en_el_10_mejor': float(np.mean(np.asarray(yp)[top]))}

## 3. Entrenar y registrar en MLflow
Una ejecución de MLflow por modelo, con sus parámetros y métricas.

In [ ]:
mlflow = None
if USAR_MLFLOW:
    import mlflow
    import mlflow.sklearn
    mlflow.set_tracking_uri('sqlite:///' + (MODELOS / 'mlflow.db').as_posix())
    if mlflow.get_experiment_by_name('retencion-90d') is None:
        mlflow.create_experiment('retencion-90d', artifact_location=(MODELOS / 'mlartifacts').resolve().as_uri())
    mlflow.set_experiment('retencion-90d')

resultados, ajustados = {}, {}
for nombre, modelo in modelos().items():
    modelo.fit(X['train'], y['train'])
    metricas = {p: evaluar(modelo, X[p], y[p]) for p in ('train', 'valid', 'test')}
    resultados[nombre], ajustados[nombre] = metricas, modelo
    print(f'{nombre:26s} AUC valid {metricas["valid"]["auc"]:.3f} | test {metricas["test"]["auc"]:.3f}'
          f' | precisión media test {metricas["test"]["precision_media"]:.3f}')
    if mlflow:
        with mlflow.start_run(run_name=nombre):
            mlflow.log_param('modelo', nombre)
            mlflow.log_param('ventana_objetivo_dias', VENTANA_OBJETIVO_DIAS)
            mlflow.log_param('n_variables', len(FEATURES))
            mlflow.log_param('huella_datos', huella_datos(df))
            for p, m in metricas.items():
                for k, v in m.items():
                    mlflow.log_metric(f'{p}_{k}', v)

## 4. Comparar con la validación
**Cómo leerlo.** `auc` mide si el modelo ordena bien a los clientes (0,5 = azar, 1 = perfecto). `precision_media` debe compararse con `tasa_base`.
`aciertos_en_el_10_mejor` es el porcentaje de clientes que **sí compraron** entre el 10 % al que el modelo daba más probabilidad: es la métrica que entendería el negocio.

In [ ]:
filas = [{'modelo': n, 'particion': p, **{k: round(v, 3) for k, v in m[p].items()}} for n, m in resultados.items() for p in ('train', 'valid', 'test')]
tabla = pd.DataFrame(filas)
tabla[tabla['particion'].isin(['valid', 'test'])].pivot(index='modelo', columns='particion', values=['auc', 'precision_media'])

In [ ]:
tabla[tabla['particion'] == 'test'][['modelo', 'tasa_base', 'aciertos_en_el_10_mejor']]

## 5. Elegir el mejor con la validación, registrarlo y guardar los artefactos
Lo que necesita la API en S2: el modelo (con su preprocesamiento) y sus metadatos.

In [ ]:
mejor = max((n for n in resultados if n != 'referencia_sin_aprender'), key=lambda n: resultados[n]['valid']['auc'])
modelo = ajustados[mejor]
print('Mejor modelo según la validación:', mejor)

if mlflow:
    with mlflow.start_run(run_name=f'MEJOR-{mejor}'):
        mlflow.log_param('modelo', mejor)
        mlflow.log_param('huella_datos', huella_datos(df))
        for p, m in resultados[mejor].items():
            for k, v in m.items():
                mlflow.log_metric(f'{p}_{k}', v)
        firma = mlflow.models.infer_signature(X['train'].head(200), modelo.predict_proba(X['train'].head(200))[:, 1])
        # cloudpickle evita la verificación de tipos de skops que MLflow 3 aplica a los árboles de scikit-learn.
        mlflow.sklearn.log_model(modelo, name='modelo', signature=firma, serialization_format='cloudpickle', input_example=X['train'].head(3))

joblib.dump(modelo, MODELOS / 'modelo.joblib')
metadatos = {
    'modelo': mejor,
    'objetivo': 'el cliente compra al menos una vez en los siguientes %d dias' % VENTANA_OBJETIVO_DIAS,
    'variables': FEATURES,
    'cortes': CORTES,
    'filas': {k: int(len(v)) for k, v in partes.items()},
    'huella_datos': huella_datos(df),
    'entrenado_en': datetime.now().isoformat(timespec='seconds'),
    'versiones': {'python': sys.version.split()[0], 'scikit-learn': sklearn.__version__, 'pandas': pd.__version__,
                  'numpy': np.__version__, 'plataforma': platform.system()},
}
(MODELOS / 'metadatos.json').write_text(json.dumps(metadatos, indent=2, ensure_ascii=False), encoding='utf-8')
(REPORTES / 'metricas.json').write_text(json.dumps({'mejor': mejor, 'modelos': resultados}, indent=2), encoding='utf-8')
metadatos

## 6. ¿Qué variables importan?
Importancia por permutación sobre la **validación**: cuánto empeora el AUC si se desordena cada variable.

In [ ]:
import matplotlib
matplotlib.use('Agg') if os.environ.get('CURSO_SIN_PANTALLA') else None
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

imp = permutation_importance(modelo, X['valid'], y['valid'], scoring='roc_auc', n_repeats=5, random_state=0)
importancia = pd.Series(imp.importances_mean, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
importancia.plot.barh(ax=ax, title='Importancia por permutación (caída del AUC en validación)')
plt.tight_layout()
fig.savefig(REPORTES / 'figures' / 'importancia_variables.png', dpi=110)
plt.show()
importancia.sort_values(ascending=False).round(4)

In [ ]:
runs = mlflow.search_runs(experiment_names=['retencion-90d']) if mlflow else None
con.close()
runs[['tags.mlflow.runName', 'metrics.valid_auc', 'metrics.test_auc', 'params.huella_datos']].round(3) if mlflow else None

### Ver MLflow en el navegador
En una terminal, desde la carpeta del proyecto:
```
mlflow ui --backend-store-uri sqlite:///models/mlflow.db
```
y abre http://127.0.0.1:5000 para comparar las ejecuciones.

## Para practicar (tarea de S1)
Con la variable nueva que añadiste a FS: reconstruye ML, vuelve a entrenar y **compara en MLflow** dos ejecuciones (antes y después). ¿Mejoró el AUC de validación? Explica en una decisión.